In [2]:
## Import Required Libraries

!pip install numpy
!pip install pandas
print("done")

done


In [3]:
import tensorflow as tf
import cv2
print(tf.__version__)
print(cv2.__version__)


C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


2.20.0
4.12.0


In [4]:
# Dataset structure
# mask_dataset/
# ├── train/
# │   ├── with_mask/
# │   └── without_mask/
# └── test/
#     ├── with_mask/
#     └── without_mask/


In [5]:
## Load Dataset

train_path = r"C:\Users\DELL\OneDrive\Desktop\mask\train"
test_path  = r"C:\Users\DELL\OneDrive\Desktop\mask\test"


In [6]:
#importing some libraires' 
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense




In [7]:
#Image Data Preprocessing

train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

# Load Training and Testing Data
train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(96,96),
    batch_size=32,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(96,96),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)


Found 1314 images belonging to 2 classes.
Found 194 images belonging to 2 classes.


In [8]:
#Build CNN Model Architecture

model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=(96,96,3)),
    MaxPooling2D(2,2),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])


C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
#Compile the Model

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [10]:
print("Train samples:", train_data.samples)
print("Test samples:", test_data.samples)


Train samples: 1314
Test samples: 194


In [11]:
import os

print(os.listdir(train_path))


['without_mask', 'with_mask']


In [12]:
# Model Evaluation

steps_per_epoch = train_data.samples // train_data.batch_size
validation_steps = test_data.samples // test_data.batch_size

history = model.fit(
    train_data,
    steps_per_epoch=steps_per_epoch,
    epochs=5,
    validation_data=test_data,
    validation_steps=validation_steps
)


Epoch 1/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.6505 - loss: 0.6383 - val_accuracy: 0.8438 - val_loss: 0.3342
Epoch 2/5
 1/41 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - accuracy: 0.9688 - loss: 0.2501

C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9688 - loss: 0.2501 - val_accuracy: 0.8385 - val_loss: 0.3754
Epoch 3/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 17s 399ms/step - accuracy: 0.8814 - loss: 0.2928 - val_accuracy: 0.9375 - val_loss: 0.1701
Epoch 4/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9375 - loss: 0.1619 - val_accuracy: 0.9427 - val_loss: 0.1726
Epoch 5/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 20s 476ms/step - accuracy: 0.9532 - loss: 0.1357 - val_accuracy: 0.9323 - val_loss: 0.1620


In [13]:
#Cheaking accuracy 

loss, acc = model.evaluate(test_data)
print("Test Accuracy:", round(acc*100, 2), "%")


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 162ms/step - accuracy: 0.9330 - loss: 0.1608
Test Accuracy: 93.3 %


In [14]:
# Predicting on new images 

from tensorflow.keras.preprocessing import image
import numpy as np

def predict_image(img_path):
    img = image.load_img(img_path, target_size=(96,96))
    img = image.img_to_array(img)/255.0
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img, verbose=0)

    if pred[0][0] > 0.5:
        print(img_path, "➡ no Mask ❌")
    else:
        print(img_path, "➡ With Mask 😷")


In [15]:
#result
predict_image(r"C:\Users\DELL\OneDrive\Desktop\mask\train\with_mask\9-with-mask.jpg")
predict_image (r"C:\Users\DELL\OneDrive\Desktop\mask\train\without_mask\13.jpg")


C:\Users\DELL\OneDrive\Desktop\mask\train\with_mask\9-with-mask.jpg ➡ With Mask 😷
C:\Users\DELL\OneDrive\Desktop\mask\train\without_mask\13.jpg ➡ no Mask ❌


In [16]:
# Save the Model

model.save("mask_detector_model.h5")
print("Model saved successfully")


Model saved successfully


In [17]:
#Load Trained Face Mask Detection Model

model = load_model("mask_detector_model.h5")

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)


In [18]:
#Testing camera

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow("Camera Test", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
# Real-Time Face Mask Detection Using Webcam 

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=3,
        minSize=(60, 60)
    )

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)


        # Safety check
        if face.size == 0:
            continue

        # Preprocess face for CNN
        face = cv2.resize(face, (96,96))
        face = face / 255.0
        face = np.reshape(face, (1,96,96,3))

        # Predict mask
        pred = model.predict(face, verbose=0)

        if pred[0][0] > 0.4:
            label = "no MASK"
            color = (0,255,0)
        else:
            label = " MASK"
            color = (0,0,255)

        # Draw results
        cv2.rectangle(frame, (x,y), (x+w,y+h), color, 2)
        cv2.putText(frame, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Mask Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
